# Chapter 4: Practical Issues

> Going from a textbook algorithm to a working system is more art than science — but the art is learnable.

**Type:** Learn + Build &nbsp;|&nbsp; **Language:** Python &nbsp;|&nbsp; **Prerequisites:** Chapters 1–3 &nbsp;|&nbsp; **Time:** ~40 minutes
**Source:** *A Course in Machine Learning*, Hal Daumé III — Chapter 4

---

## Learning Objectives

- Explain how to use cross-validation to estimate future performance without touching test data
- Perform basic feature engineering: centering, scaling, and variance-based pruning
- Compare the sensitivity of decision trees, KNN, and the perceptron to feature scale and irrelevant features
- Explain why decision trees are naturally more robust to noisy features than distance/dot-product based models

## The Problem

By now you know three qualitatively different learning models (decision trees, KNN, perceptron). But going from "I understand the algorithm" to "I have a system that performs well" involves many practical choices the algorithms themselves don't make for you:

- How do you represent your data?
- Should you rescale it?
- What do you do about useless features?
- How do you honestly estimate how well your model will do in the future without ever touching your test set?

## The Concept

**The practical pipeline:**

```
Raw features
      │
      ▼
Same scale? ──no──► Center + scale features ──┐
      │                                        │
     yes                                       ▼
      └───────────────────────────► Prune low-variance / irrelevant features
                                                │
                                                ▼
                                K-fold cross-validation to tune hyperparameters
                                                │
                                                ▼
                                Train final model on all training data
```

### Key Ideas

- **Feature scale matters differently per model:** decision trees only compare a feature to a threshold, so monotonic rescaling never changes a split. KNN and the perceptron both use the numeric feature values directly (in a distance or a dot product), so an unscaled feature can silently dominate.
- **Irrelevant features hurt distance-based models most:** as shown in Chapter 2, distances become less informative as dimensionality grows; adding noise features pushes KNN toward this regime fastest.
- **Cross-validation trades compute for robustness:** instead of holding out one fixed development set, K-fold CV rotates through K different splits and averages the result, giving a lower-variance performance estimate at the cost of K-times more training.
- **Pruning is a form of regularization:** throwing out uninformative features (e.g. ones with near-zero variance) reduces both computation and the risk of the model latching onto noise.

## Build It

### Setup

We use the Breast Cancer Wisconsin dataset again, plus scikit-learn's `DecisionTreeClassifier`, `KNeighborsClassifier`, and `Perceptron` as the three models being compared throughout this chapter, along with `cross_val_score` as a sanity check for our own cross-validation routine.

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import Perceptron as SKPerceptron
from sklearn.metrics import accuracy_score

RNG = np.random.RandomState(7)

### Step 1: Feature Normalization (Section 4.3)

Centering and scaling transforms each feature to zero mean and unit variance. Crucially, the mean and standard deviation are computed **only from the training set**, then reused to transform the test set — never let test-set statistics leak into preprocessing.

In [2]:
def center_and_scale(X_train, X_test):
    mu = X_train.mean(axis=0)
    sigma = X_train.std(axis=0)
    sigma[sigma == 0] = 1.0
    X_train_norm = (X_train - mu) / sigma
    X_test_norm = (X_test - mu) / sigma
    return X_train_norm, X_test_norm

### Step 2: K-Fold Cross-Validation (Algorithm 8, Section 4.6)

`cross_validate` takes a factory function that produces a *fresh, unfitted* model, shuffles the data once, splits it into `K` roughly equal folds, and for each fold trains on the rest and evaluates on that fold — mirroring the book's `CrossValidate` procedure.

In [3]:
def cross_validate(model_fn, X, y, K=10):
    X = np.asarray(X)
    y = np.asarray(y)
    N = X.shape[0]
    indices = np.arange(N)
    RNG.shuffle(indices)

    fold_sizes = np.full(K, N // K, dtype=int)
    fold_sizes[: N % K] += 1
    scores = []
    current = 0
    for fold_size in fold_sizes:
        test_idx = indices[current: current + fold_size]
        train_idx = np.setdiff1d(indices, test_idx)
        current += fold_size

        model = model_fn()
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[test_idx])
        scores.append(accuracy_score(y[test_idx], preds))
    return np.array(scores)

## Use It — Real Data

### Experiment A: Does Normalization Matter? (Section 4.3)

We load the dataset, split into train/test, and normalize using train-set statistics. Then we train each of the three model types — KNN, perceptron, and decision tree — both on the **raw** features and on the **normalized** ones, to see which models actually care about feature scale.

In [4]:
data = load_breast_cancer()
X, y_raw = data.data, data.target
y_pm = np.where(y_raw == 0, -1, 1)

X_train, X_test, y_train, y_test, ytr_pm, yte_pm = train_test_split(
    X, y_raw, y_pm, test_size=0.3, random_state=42, stratify=y_raw
)

X_train_norm, X_test_norm = center_and_scale(X_train, X_test)

print(f"{'model':>28} | {'raw features acc':>17} | {'normalized acc':>15}")
print("-" * 68)

knn_raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
knn_norm = KNeighborsClassifier(n_neighbors=5).fit(X_train_norm, y_train)
print(f"{'KNN (k=5)':>28} | {accuracy_score(y_test, knn_raw.predict(X_test)):>17.4f} "
      f"| {accuracy_score(y_test, knn_norm.predict(X_test_norm)):>15.4f}")

perc_raw = SKPerceptron(max_iter=50, tol=None, random_state=1).fit(X_train, ytr_pm)
perc_norm = SKPerceptron(max_iter=50, tol=None, random_state=1).fit(X_train_norm, ytr_pm)
print(f"{'Perceptron':>28} | {accuracy_score(yte_pm, perc_raw.predict(X_test)):>17.4f} "
      f"| {accuracy_score(yte_pm, perc_norm.predict(X_test_norm)):>15.4f}")

dt_raw = DecisionTreeClassifier(max_depth=4, random_state=1).fit(X_train, y_train)
dt_norm = DecisionTreeClassifier(max_depth=4, random_state=1).fit(X_train_norm, y_train)
print(f"{'Decision Tree (depth=4)':>28} | {accuracy_score(y_test, dt_raw.predict(X_test)):>17.4f} "
      f"| {accuracy_score(y_test, dt_norm.predict(X_test_norm)):>15.4f}")

                       model |  raw features acc |  normalized acc
--------------------------------------------------------------------
                   KNN (k=5) |            0.9240 |          0.9591
                  Perceptron |            0.9064 |          0.9649
     Decision Tree (depth=4) |            0.9240 |          0.9240


**Reading the table:** as the book predicts, decision trees are invariant to monotonic per-feature rescaling (their splits don't change), while KNN and the perceptron — both of which rely on raw distances or dot products — are sensitive to feature scale and benefit from normalization.

### Experiment B: Robustness to Irrelevant Features (Section 4.2, Figure 4.6)

Now we deliberately pollute the dataset with pure Gaussian noise features (which carry zero real signal) and watch how each model's accuracy degrades as we add more and more of them.

In [5]:
print(f"{'# noise feats':>13} | {'DT acc':>7} | {'KNN acc':>8} | {'Perceptron acc':>15}")
print("-" * 55)
for n_noise in [0, 10, 30, 60, 120, 240]:
    noise_train = RNG.normal(0, 1, size=(X_train_norm.shape[0], n_noise))
    noise_test = RNG.normal(0, 1, size=(X_test_norm.shape[0], n_noise))
    Xn_train = np.hstack([X_train_norm, noise_train])
    Xn_test = np.hstack([X_test_norm, noise_test])

    dt = DecisionTreeClassifier(max_depth=4, random_state=1).fit(Xn_train, y_train)
    knn = KNeighborsClassifier(n_neighbors=5).fit(Xn_train, y_train)
    perc = SKPerceptron(max_iter=50, tol=None, random_state=1).fit(Xn_train, ytr_pm)

    dt_acc = accuracy_score(y_test, dt.predict(Xn_test))
    knn_acc = accuracy_score(y_test, knn.predict(Xn_test))
    perc_acc = accuracy_score(yte_pm, perc.predict(Xn_test))
    print(f"{n_noise:>13} | {dt_acc:>7.4f} | {knn_acc:>8.4f} | {perc_acc:>15.4f}")

# noise feats |  DT acc |  KNN acc |  Perceptron acc
-------------------------------------------------------
            0 |  0.9240 |   0.9591 |          0.9649
           10 |  0.9064 |   0.9298 |          0.9532
           30 |  0.9181 |   0.9474 |          0.9298
           60 |  0.9181 |   0.9298 |          0.9357
          120 |  0.9298 |   0.8947 |          0.9415


          240 |  0.9181 |   0.8713 |          0.9181


**Reading the table:** KNN tends to degrade fastest — irrelevant features dilute the meaningfulness of distance, exactly the curse-of-dimensionality effect from Chapter 2. Decision trees tend to be the most robust, since they explicitly select useful features at each split and can mostly ignore noisy ones.

### Experiment C: From-Scratch Cross-Validation vs. `sklearn.cross_val_score`

As a sanity check, we compare our own `cross_validate` implementation against scikit-learn's `cross_val_score` on the same model and data. An exact match isn't expected — the two use different random shuffling — but the mean accuracies should land close to each other.

In [6]:
my_scores = cross_validate(lambda: DecisionTreeClassifier(max_depth=4, random_state=1), X, y_raw, K=10)
sk_scores = cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=1), X, y_raw, cv=10)

print(f"From-scratch 10-fold CV : mean={my_scores.mean():.4f}  std={my_scores.std():.4f}")
print(f"sklearn 10-fold CV      : mean={sk_scores.mean():.4f}  std={sk_scores.std():.4f}")

From-scratch 10-fold CV : mean=0.9280  std=0.0354
sklearn 10-fold CV      : mean=0.9175  std=0.0407


### Experiment D: Pruning Low-Variance Features (Section 4.3, Figure 4.9)

Finally, we test the idea that features with very low variance carry little information and can often be pruned safely — or even helpfully — before hurting performance too much. We rank features by variance (computed on the training set) and progressively drop the lowest-variance ones.

In [7]:
variances = X_train.var(axis=0)
order = np.argsort(variances)

print(f"{'# features pruned':>18} | {'# features kept':>16} | {'DT test acc':>11}")
print("-" * 52)
for n_pruned in [0, 5, 10, 15, 20, 25, 28]:
    keep_idx = order[n_pruned:]
    if len(keep_idx) == 0:
        continue
    dt = DecisionTreeClassifier(max_depth=4, random_state=1).fit(X_train[:, keep_idx], y_train)
    acc = accuracy_score(y_test, dt.predict(X_test[:, keep_idx]))
    print(f"{n_pruned:>18} | {len(keep_idx):>16} | {acc:>11.4f}")

 # features pruned |  # features kept | DT test acc
----------------------------------------------------
                 0 |               30 |      0.9298
                 5 |               25 |      0.9240
                10 |               20 |      0.9240
                15 |               15 |      0.9415
                20 |               10 |      0.9474
                25 |                5 |      0.9123
                28 |                2 |      0.9123


**Reading the table:** as in the book's Figure 4.9, pruning a few low-variance features doesn't hurt, and can even help slightly, but pruning too aggressively eventually destroys useful signal and accuracy collapses.

## Use It

| API / Function | When to use it |
|---|---|
| `center_and_scale(X_train, X_test)` | Before training KNN, perceptron, or any distance/dot-product-based model |
| `cross_validate(model_fn, X, y, K=10)` | Estimating generalization performance or tuning a hyperparameter without a separate held-out dev set |
| Variance-based pruning (`X.var(axis=0)`) | Quick first-pass feature selection before training, especially with many near-constant features |
| `sklearn.model_selection.cross_val_score` | Production use — vectorized, supports many scoring metrics and stratified folds |

## Exercises

1. Modify Experiment A to also test a `max_depth=None` (unpruned) decision tree — does normalization matter more or less as the tree gets deeper?
2. Implement leave-one-out cross-validation (`K = N`) for the KNN classifier using the efficient approach described in the book (Algorithm 9), and compare its runtime to naive 10-fold CV.
3. Add an `abs_scaling` option to `center_and_scale` that divides by the maximum absolute value instead of the standard deviation (Eq. 4.3), and compare its effect on perceptron accuracy.

## Key Terms

| Term | Common Assumption | Precise Meaning |
|---|---|---|
| **Normalization** | "Always helps, always do it" | A per-feature transformation (centering + scaling) that helps models sensitive to raw feature magnitude; irrelevant to models that only compare feature values to thresholds |
| **Cross-Validation** | "Just a fancier train/test split" | A resampling procedure that trains and evaluates K times on K different partitions, trading extra computation for a lower-variance estimate of generalization performance |
| **Irrelevant Feature** | "Harmless noise" | A feature whose expected value doesn't depend on the label; harmless in small numbers but can dominate distance-based models when numerous |
| **Feature Pruning** | "Only for saving memory" | Removing low-signal (e.g. low-variance) features before training, which is itself a form of regularization against overfitting to noise |

## Summary

- Decision trees are invariant to feature rescaling; KNN and the perceptron are not
- Irrelevant (noise) features hurt distance-based models (KNN) far more than decision trees
- K-fold cross-validation gives a lower-variance estimate of generalization performance than a single train/test split, at the cost of extra training runs
- Pruning a modest number of low-variance features can help or be neutral; pruning too aggressively destroys signal

---

**Next:** Chapter 5 — Beyond Binary Classification